# Current "Official" Preprocessing Implementation

In [43]:
from pathlib import Path
import polars as pl

COLUMNS_TO_KEEP = (
    "author.username",
    "author.avatar",
    "attachments.0.url",
    "attachments.1.url",
    "attachments.2.url",
    "attachments.3.url",
    "attachments.4.url",
    "attachments.5.url",
    "attachments.6.url",
    "attachments.7.url",
    "attachments.8.url",
    "reactions.0.emoji.name",
    "reactions.1.emoji.name",
    "reactions.2.emoji.name",
    "reactions.3.emoji.name",
    "reactions.4.emoji.name",
    "reactions.5.emoji.name",
    "content",
    "timestamp",
)

UID_TO_UNAME = {
    "1211781489931452447": "Wordle",
    "329382986405642240": "evandabest_",
    "1416259149997670433": "joeisaloserandnotgettingoo_34014",
    "518633088180420609": "kingwooziee",
    "834249758159798273": "mango2875",
    "603328024263262208": "oof123",
    "179745514080829440": "plazmama",
    "458421510596460554": "rbtro",
    "511655545321685019": "vhazey",
    "723005048040325140": "whorsey.",
    "601380778684841994": "yannaner",
}

data_dir = Path("crazy_gc_data")
csv_paths = sorted(data_dir.rglob("*.csv"))

if not csv_paths:
    raise FileNotFoundError(f"No CSV files under {data_dir.resolve()}")

# Each page can export a different column set → multi-file scan_csv fails with
# "schema lengths differ". Read per file and stack with relaxed alignment.
dfs = [
    pl.read_csv(
        p,
        infer_schema_length=100_000,
        try_parse_dates=True,
    )
    for p in csv_paths
]

reduced_parts: list[pl.DataFrame] = []
for df in dfs:
    present = [c for c in COLUMNS_TO_KEEP if c in df.columns]
    reduced_parts.append(df.select(present))

# Different pages omit columns → different widths. `vertical_relaxed` can still
# error ("schema lengths differ") here; `diagonal` unions columns, null for missing.
reduced_df = pl.concat(reduced_parts, how="diagonal")

# print(f"Loaded {len(csv_paths)} file(s), {reduced_df.height} rows × {reduced_df.width} columns")
# print(reduced_df.head(5))
reduced_df.write_parquet("discord_gc_combined.parquet") # Recording raw dataset for later use(Committed to github for sharing across my pcs)
print(reduced_df.select(["author.username", "content"]).head(10))

# All message bodies (non-empty text). Re-run the CSV→parquet cell after adding `content` to COLUMNS_TO_KEEP.
df = pl.read_parquet("discord_gc_combined.parquet").filter(
        pl.col("content").is_not_null()
        & (pl.col("content").str.strip_chars().str.len_chars() > 0)
    )

df = df.with_columns(
    pl.col("content").str.replace_many(
        list(UID_TO_UNAME.keys()),
        list(UID_TO_UNAME.values()),
    )
)

with open("messages.txt", "w") as msgs_file:
    for row in df.iter_rows(named=True):
        msgs_file.write(f"{row["author.username"]}: {row["content"]}\n---\n")

shape: (10, 2)
┌─────────────────┬─────────────────────────────────┐
│ author.username ┆ content                         │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ rbtro           ┆ https://tenor.com/view/wash-mo… │
│ evandabest_     ┆ was the other one nuked         │
│ oof123          ┆ https://tenor.com/sTFz.gif      │
│ evandabest_     ┆ https://tenor.com/view/attack-… │
│ plazmama        ┆ What happened?                  │
│ mango2875       ┆ what happened to the old one😭  │
│ rbtro           ┆ dw bout it                      │
│ mango2875       ┆ what were yall talking abt bro… │
│ mango2875       ┆ gg                              │
│ evandabest_     ┆ we were due for a refresh       │
└─────────────────┴─────────────────────────────────┘


# Dataset experimantation

In [53]:
# ALL attachements to go! They are not accissible since they are private
# Call Parcitipants to go

df = pl.read_parquet("discord_gc_combined.parquet")
print(df.filter(pl.col("author.avatar").is_not_null()).select("author.avatar").to_numpy()[0])
names = df.select("author.username").to_numpy().tolist()
names = [name[0] for name in names]
for name in set(names):
    print(name)


['95b91af482446975e630077533da0f4b']
evandabest_
whorsey.
vhazey
Wordle
joeisaloserandnotgettingoo_34014
kingwooziee
rbtro
plazmama
yannaner
mango2875
oof123


In [ ]:
# Unique (username, Discord user id) pairs from raw exports (author.id is not in the parquet slice).
from pathlib import Path
import polars as pl

data_dir = Path("crazy_gc_data")
csv_paths = sorted(data_dir.rglob("*.csv"))
if not csv_paths:
    raise FileNotFoundError(f"No CSV files under {data_dir.resolve()}")

parts: list[pl.DataFrame] = []
for p in csv_paths:
    df = pl.read_csv(p, infer_schema_length=100_000, try_parse_dates=True)
    if "author.username" in df.columns and "author.id" in df.columns:
        parts.append(
            df.select(
                pl.col("author.username").cast(pl.Utf8),
                pl.col("author.id").cast(pl.Utf8),
            )
        )

if not parts:
    raise ValueError("No CSV contained both author.username and author.id")

stacked = pl.concat(parts, how="diagonal")
username_id = (
    stacked.drop_nulls()
    .unique()
    .sort("author.username")
)
print(username_id.write_csv())
print(f"\n{username_id.height} distinct username–id pair(s)")

author.username,author.id
Wordle,1211781489931452447
evandabest_,329382986405642240
joeisaloserandnotgettingoo_34014,1416259149997670433
kingwooziee,518633088180420609
mango2875,834249758159798273
oof123,603328024263262208
plazmama,179745514080829440
rbtro,458421510596460554
vhazey,511655545321685019
whorsey.,723005048040325140
yannaner,601380778684841994


11 distinct username–id pair(s)


In [20]:
from pathlib import Path
import polars as pl

csv_path = sorted(Path("crazy_gc_data").rglob("*.csv"))[0]
df = pl.read_csv(csv_path, infer_schema_length=100_000, try_parse_dates=True)
out = (
    df.select("author.username", "referenced_message.content")
    .filter(
        pl.col("referenced_message.content").is_not_null()
        & (pl.col("referenced_message.content").str.strip_chars().str.len_chars() > 0)
    )
)
print(out)

shape: (4_237, 2)
┌─────────────────┬─────────────────────────────────┐
│ author.username ┆ referenced_message.content      │
│ ---             ┆ ---                             │
│ str             ┆ str                             │
╞═════════════════╪═════════════════════════════════╡
│ mango2875       ┆ u will have sum to lose         │
│ evandabest_     ┆ expectations                    │
│ plazmama        ┆ "Soft soft soft soft soft hard… │
│ mango2875       ┆ lmao she unfollowed me on inst… │
│ kingwooziee     ┆ https://tenor.com/view/no-evid… │
│ …               ┆ …                               │
│ rbtro           ┆ CFA mfs seeing these comments … │
│ oof123          ┆ <@603328024263262208> r u comi… │
│ whorsey.        ┆ https://www.cbsnews.com/news/h… │
│ mango2875       ┆ Someone send address            │
│ mango2875       ┆ Just come to the eats ngl       │
└─────────────────┴─────────────────────────────────┘
